# Analysis 1A extension — nearest response-cell similarity

# Analysis 1A extension: observed-DGE response similarity

This notebook implements a **post-hoc diagnostic analysis** of unseen-cell transferability.

It uses observed differential gene expression (DGE) from the held-out test cell to quantify its response similarity to cells in the training set. Consequently, this similarity is **not available prospectively for a truly unmeasured cell** and must not be presented as an input to DEPICT or as a deployable selection criterion. Its purpose is to test whether difficult held-out cells have atypical perturbational response programs despite having similar untreated baseline states.

Core design choices:

- DGE is reconstructed after the same sample-wise total normalization used for DEPICT.
- Replicate profiles are averaged within each exact `(drug, dose, duration)` condition and cell.
- Only training–test cell pairs sharing at least **30 exact perturbation conditions** are eligible.
- The held-out cell is the unit of downstream association analysis.
- The same four cell-level performance outcomes and the same within-split Spearman/bootstrap framework as Analysis 1A are used.

**Audited revision v2:** fixes condition-metadata construction, uses `pert_id` for exact compound matching when available, validates split membership and paired controls, and requires at least 30 usable matched correlations for the mean-condition analysis.

**Audited revision v3:** exact condition matching now uses `(pert_id, rounded dose, rounded duration)` tuple keys rather than formatted strings, preventing false duplicate-condition collisions.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import scanpy as sc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Configuration

In [ ]:

# -----------------------------------------------------------------------------
# Configuration — edit paths if your project location differs.
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path("~/DEPICT")
MAIN_WORK_DIR = PROJECT_ROOT / "Code/downstream_analysis_code/TransferabilityUnseenCell"


CONFIG = {
    "adata_path": PROJECT_ROOT / "Data/FinalData/adataAfterClean.h5ad",
    "prediction_root": MAIN_WORK_DIR / "predicted_dge",
    "a1_primary_metrics_path": (
        MAIN_WORK_DIR / "a1_ood_similarity"
        / "tables" / "a1_primary_cell_level_metrics.csv"
    ),
    "output_dir": MAIN_WORK_DIR / "a1_response_similarity_mean_condition_v3",
    "cell_splits": [f"cell_split{i}" for i in range(1, 6)],
    "expected_n_genes": 978,

    # Use the stable perturbation identifier for exact cross-cell matching.
    # pert_iname is retained only as a readable annotation when available.
    "drug_id_column": "pert_id",
    "drug_name_column": "pert_iname",

    "dose_column": "dose",
    "time_column": "pert_time",
    "cell_column": "cell_id",
    "control_column": "control",
    "paired_control_column": "paired_control_index",

    # A test-training cell pair must contribute at least this many usable,
    # exactly matched drug-dose-duration conditions.
    "min_shared_perturbations": 30,

    # Exact matching after rounding only to absorb floating-point storage noise.
    "dose_round_decimals": 6,
    "time_round_decimals": 6,

    "n_bootstrap": 2000,
    "random_seed": 66,
    "epsilon": 1e-8,

    # Condition-level output can be large in the mean-condition analysis.
    "save_condition_level_csv": False,
}

OUT = Path(CONFIG["output_dir"])
TABLE_DIR = OUT / "tables"
MANIFEST_DIR = OUT / "manifests"
for directory in (OUT, TABLE_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG


{'adata_path': PosixPath('/work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/data/whole_data_Jul11/adataAfterClean.h5ad'),
 'prediction_root': PosixPath('/work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/predicted_dge'),
 'a1_primary_metrics_path': PosixPath('/work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/a1_ood_similarity/tables/a1_primary_cell_level_metrics.csv'),
 'output_dir': PosixPath('/work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/a1_response_similarity_mean_condition_v3'),
 'cell_splits': ['cell_split1',
  'cell_split2',
  'cell_split3',
  'cell_split4',
  'cell_split5'],
 'expected_n_genes': 978,
 'drug_id_column': 'pert_id',
 'drug_name_column': 'pert_iname',
 'dose_column': 'dose',
 'time_column': 'pert_time',
 'cell_column': 'cell_id',
 'con

## 2. Reproducibility and statistical helpers

In [3]:
# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------
def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def save_table(df: pd.DataFrame, stem: str, index: bool = False, save_csv: bool = True) -> None:
    if save_csv:
        df.to_csv(TABLE_DIR / f"{stem}.csv", index=index)
    try:
        df.to_parquet(TABLE_DIR / f"{stem}.parquet", index=index)
    except Exception as exc:
        print(f"Parquet not written for {stem}: {exc!r}")


def decode_h5_strings(values):
    values = np.asarray(values)
    if values.dtype.kind == "S":
        return np.char.decode(values, "utf-8")
    if values.dtype.kind == "O":
        return np.asarray([
            x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)
            for x in values
        ], dtype=object)
    return values.astype(str)


def safe_pearson(x, y, eps: float = 1e-12) -> float:
    x = np.asarray(x, dtype=np.float64).ravel()
    y = np.asarray(y, dtype=np.float64).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < 2:
        return np.nan
    x = x - x.mean()
    y = y - y.mean()
    denom = np.sqrt(np.dot(x, x) * np.dot(y, y))
    if denom <= eps:
        return np.nan
    return float(np.dot(x, y) / denom)


def safe_spearman(x, y, min_n: int = 3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan, np.nan
    result = spearmanr(x, y)
    return float(result.statistic), float(result.pvalue)


def ols_slope_intercept(x, y, min_n: int = 3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x, y, 1)
    fitted = intercept + slope * x
    ss_res = np.sum((y - fitted) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = np.nan if ss_tot <= 0 else 1 - ss_res / ss_tot
    return float(slope), float(intercept), float(r2)


def percentile_ci(values, alpha: float = 0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return tuple(np.quantile(values, [alpha / 2, 1 - alpha / 2]))


def deterministic_seed(*parts) -> int:
    text = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(text).digest()[:8], "little") % (2**32 - 1)


def bootstrap_fold_association(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    n_bootstrap: int,
    seed: int,
):
    sub = df[[x_col, y_col]].dropna().reset_index(drop=True)
    x = sub[x_col].to_numpy(float)
    y = sub[y_col].to_numpy(float)
    n = len(sub)

    rho, p = safe_spearman(x, y)
    slope, intercept, r2 = ols_slope_intercept(x, y)

    rng = np.random.default_rng(seed)
    boot_rho, boot_slope = [], []
    if n >= 3:
        for _ in range(n_bootstrap):
            ix = rng.integers(0, n, size=n)
            r, _ = safe_spearman(x[ix], y[ix])
            s, _, _ = ols_slope_intercept(x[ix], y[ix])
            boot_rho.append(r)
            boot_slope.append(s)

    rho_lo, rho_hi = percentile_ci(boot_rho)
    slope_lo, slope_hi = percentile_ci(boot_slope)
    return {
        "n_cells": n,
        "n_bootstrap_requested": int(n_bootstrap),
        "n_bootstrap_estimable_rho": int(np.isfinite(boot_rho).sum()),
        "n_bootstrap_estimable_slope": int(np.isfinite(boot_slope).sum()),
        "spearman_rho": rho,
        "spearman_p_descriptive": p,
        "spearman_bootstrap_ci_low": rho_lo,
        "spearman_bootstrap_ci_high": rho_hi,
        "ols_slope": slope,
        "ols_intercept": intercept,
        "ols_r2": r2,
        "ols_slope_bootstrap_ci_low": slope_lo,
        "ols_slope_bootstrap_ci_high": slope_hi,
        "association_estimable": bool(np.isfinite(rho)),
    }

## 3. Reconstruct observed DGE

In [4]:

# -----------------------------------------------------------------------------
# Load AnnData and reconstruct observed DGE for all non-control rows.
# -----------------------------------------------------------------------------
adata_path = Path(CONFIG["adata_path"])
require(adata_path.exists(), f"AnnData not found: {adata_path}")

adata = sc.read(adata_path)
require(
    adata.n_vars == CONFIG["expected_n_genes"],
    f"Expected {CONFIG['expected_n_genes']} genes, found {adata.n_vars}.",
)
require(adata.obs_names.is_unique, "AnnData obs_names must be unique.")

# Prefer pert_id for exact compound identity. Fall back to pert_iname only if
# the configured identifier is genuinely unavailable.
drug_id_col = CONFIG["drug_id_column"]
if drug_id_col not in adata.obs.columns:
    fallback = CONFIG["drug_name_column"]
    require(
        fallback in adata.obs.columns,
        f"Neither {drug_id_col!r} nor fallback {fallback!r} exists in adata.obs.",
    )
    print(
        f"WARNING: {drug_id_col!r} is unavailable; using {fallback!r} for "
        "cross-cell condition matching."
    )
    drug_id_col = fallback

required_obs = [
    drug_id_col,
    CONFIG["dose_column"],
    CONFIG["time_column"],
    CONFIG["cell_column"],
    CONFIG["control_column"],
    CONFIG["paired_control_column"],
    *CONFIG["cell_splits"],
]
missing_obs = [c for c in required_obs if c not in adata.obs.columns]
require(not missing_obs, f"AnnData is missing required obs columns: {missing_obs}")

# Audit cell-split labels before any computation.
for split_type in CONFIG["cell_splits"]:
    labels = set(adata.obs[split_type].dropna().astype(str).unique())
    require(
        {"train", "test"}.issubset(labels),
        f"{split_type} does not contain both 'train' and 'test'. Found: {sorted(labels)}",
    )

print("Applying sample-wise total normalization to match DEPICT preprocessing...")
sc.pp.normalize_total(adata)

control_series = pd.to_numeric(
    adata.obs[CONFIG["control_column"]], errors="coerce"
)
require(
    control_series.notna().all(),
    f"{CONFIG['control_column']} contains missing/non-numeric values.",
)
unexpected_controls = set(control_series.unique()).difference({0, 1})
require(
    not unexpected_controls,
    f"Unexpected control labels: {sorted(unexpected_controls)}; expected only 0/1.",
)
control_values = control_series.to_numpy()
perturbation_positions = np.flatnonzero(control_values == 0)
require(len(perturbation_positions) > 0, "No perturbation rows found.")

obs_index = pd.Index(adata.obs_names.astype(str))
paired_raw = adata.obs.iloc[perturbation_positions][CONFIG["paired_control_column"]]
require(paired_raw.notna().all(), "Some perturbation rows lack paired_control_index.")
paired_ids = paired_raw.astype(str)
paired_positions = obs_index.get_indexer(paired_ids)
require(
    (paired_positions >= 0).all(),
    "Some paired_control_index values are absent from adata.obs_names.",
)
require(
    np.all(control_values[paired_positions] == 1),
    "Some paired_control_index values do not reference control rows.",
)

def dense_rows(X, positions):
    block = X[positions]
    if hasattr(block, "toarray"):
        block = block.toarray()
    return np.asarray(block, dtype=np.float32)

print(
    f"Reconstructing observed DGE for {len(perturbation_positions):,} "
    "perturbation rows..."
)
observed_dge = dense_rows(adata.X, perturbation_positions)
observed_dge -= dense_rows(adata.X, paired_positions)
require(
    np.isfinite(observed_dge).all(),
    "Observed DGE contains non-finite values after normalization/subtraction.",
)

meta = adata.obs.iloc[perturbation_positions].copy()
meta["adata_row"] = perturbation_positions
meta["obs_name"] = adata.obs_names[perturbation_positions].astype(str)

# Preserve missingness during validation; avoid converting NaN to the literal "nan".
meta["drug_key"] = meta[drug_id_col].astype("string").str.strip()
meta["cell_id"] = meta[CONFIG["cell_column"]].astype("string").str.strip()
meta["dose_key"] = pd.to_numeric(
    meta[CONFIG["dose_column"]], errors="coerce"
).round(CONFIG["dose_round_decimals"])
meta["time_key"] = pd.to_numeric(
    meta[CONFIG["time_column"]], errors="coerce"
).round(CONFIG["time_round_decimals"])

if CONFIG["drug_name_column"] in meta.columns:
    meta["drug_name"] = (
        meta[CONFIG["drug_name_column"]].astype("string").str.strip()
    )
else:
    meta["drug_name"] = meta["drug_key"]

valid_meta = (
    meta["drug_key"].notna()
    & meta["drug_key"].ne("")
    & meta["cell_id"].notna()
    & meta["cell_id"].ne("")
    & meta["dose_key"].notna()
    & meta["time_key"].notna()
)
require(
    valid_meta.all(),
    "Missing/invalid drug identifier, dose, duration, or cell identifier "
    "among perturbation rows.",
)

meta["drug_key"] = meta["drug_key"].astype(str)
meta["drug_name"] = meta["drug_name"].fillna(meta["drug_key"]).astype(str)
meta["cell_id"] = meta["cell_id"].astype(str)

meta["condition_key"] = (
    meta["drug_key"]
    + "|dose="
    + meta["dose_key"].map(
        lambda x: f"{x:.{CONFIG['dose_round_decimals']}f}"
    )
    + "|time="
    + meta["time_key"].map(
        lambda x: f"{x:.{CONFIG['time_round_decimals']}f}"
    )
)

print(f"Drug identity column used: {drug_id_col}")
print(f"Observed-DGE matrix: {observed_dge.shape}")
print(f"Unique cells: {meta['cell_id'].nunique()}")
print(
    "Unique exact drug-dose-duration conditions: "
    f"{meta['condition_key'].nunique()}"
)


Applying sample-wise total normalization to match DEPICT preprocessing...
Reconstructing observed DGE for 836,649 perturbation rows...
Drug identity column used: pert_id
Observed-DGE matrix: (836649, 978)
Unique cells: 82
Unique exact drug-dose-duration conditions: 48680


## 4. Construct replicate-averaged cell-condition DGE centroids

In [5]:

# -----------------------------------------------------------------------------
# Aggregate replicate DGE profiles within cell × exact drug-dose-duration condition.
# This gives every distinct condition one centroid and prevents replicate-rich
# conditions from receiving greater pairwise weight.
# -----------------------------------------------------------------------------
group_cols = [
    "cell_id", "drug_key", "dose_key", "time_key", "condition_key"
]

group_index = pd.MultiIndex.from_frame(
    meta[group_cols].reset_index(drop=True),
    names=group_cols,
)
group_codes, unique_groups = pd.factorize(group_index, sort=True)
require((group_codes >= 0).all(), "Unexpected missing group code.")
n_groups = len(unique_groups)

# IMPORTANT: pandas may drop MultiIndex level names during factorization.
# Construct the metadata explicitly from tuples rather than using
# unique_groups.to_frame(), which caused the previous missing-cell_id bug.
condition_meta = pd.DataFrame(
    unique_groups.tolist(),
    columns=group_cols,
)

dge_sums = np.zeros(
    (n_groups, CONFIG["expected_n_genes"]),
    dtype=np.float64,
)
np.add.at(dge_sums, group_codes, observed_dge)

replicate_counts = np.bincount(group_codes, minlength=n_groups)
require(
    (replicate_counts > 0).all(),
    "At least one cell-condition group has zero replicates.",
)

condition_dge = (
    dge_sums / replicate_counts[:, None]
).astype(np.float32)
del dge_sums

require(
    condition_dge.shape == (n_groups, CONFIG["expected_n_genes"]),
    f"Unexpected condition-DGE shape: {condition_dge.shape}",
)
require(
    np.isfinite(condition_dge).all(),
    "Condition-level DGE centroids contain non-finite values.",
)

condition_meta["n_replicates"] = replicate_counts.astype(int)
condition_meta["condition_row"] = np.arange(n_groups, dtype=int)

require(
    condition_meta["condition_row"].is_unique,
    "condition_row must be unique.",
)
require(
    not condition_meta.duplicated(
        ["cell_id", "drug_key", "dose_key", "time_key"]
    ).any(),
    "Duplicate cell × drug × dose × duration centroids remain after aggregation.",
)

save_table(condition_meta, "condition_centroid_metadata")
np.save(OUT / "condition_dge_centroids.npy", condition_dge)

print(f"Constructed {n_groups:,} cell-condition DGE centroids.")
condition_coverage = (
    condition_meta.groupby("cell_id", as_index=False)
    .agg(
        n_conditions=("condition_key", "nunique"),
        median_replicates=("n_replicates", "median"),
        max_replicates=("n_replicates", "max"),
    )
    .sort_values("n_conditions", ascending=False)
)
save_table(condition_coverage, "cell_condition_coverage")
display(condition_coverage.head(15))


Constructed 234,270 cell-condition DGE centroids.


,cell_id,n_conditions,median_replicates,max_replicates
37,MCF7,30075,3.0,983
79,VCAP,28739,3.0,175
53,PC3,24938,3.0,937
1,A549,20452,3.0,130
0,A375,19237,3.0,670
28,HT29,18319,3.0,887
15,HA1E,13516,3.0,811
17,HCC515,8729,3.0,52
22,HEPG2,6072,3.0,44
49,NPC,4616,3.0,55


## 5. Load the original Analysis 1A performance outcomes

In [6]:
# -----------------------------------------------------------------------------
# Load the exact cell-level performance outcomes used by original Analysis 1A.
# Fallback: reconstruct them from the saved test-prediction HDF5 files.
# -----------------------------------------------------------------------------
METRIC_COLUMNS = ["delta_pcc", "delta_r2", "dge_mse", "mse_gain_over_naive"]

def safe_pearson_rows(A, B, eps=1e-8, undefined_value=0.0):
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    Ac = A - A.mean(axis=1, keepdims=True)
    Bc = B - B.mean(axis=1, keepdims=True)
    denom = np.sqrt((Ac * Ac).sum(axis=1) * (Bc * Bc).sum(axis=1))
    out = np.full(A.shape[0], float(undefined_value), dtype=float)
    valid = denom > eps
    out[valid] = (Ac[valid] * Bc[valid]).sum(axis=1) / denom[valid]
    return out

def rowwise_r2(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = ((y_true - y_pred) ** 2).sum(axis=1)
    centered = y_true - y_true.mean(axis=1, keepdims=True)
    ss_tot = (centered ** 2).sum(axis=1)
    out = np.empty(y_true.shape[0], dtype=float)
    nonconstant = ss_tot > eps
    out[nonconstant] = 1.0 - ss_res[nonconstant] / ss_tot[nonconstant]
    out[~nonconstant] = np.where(ss_res[~nonconstant] <= eps, 1.0, 0.0)
    return out

def prediction_h5_path(split_type):
    return Path(CONFIG["prediction_root"]) / split_type / "predicted_dge_test.h5"

def reconstruct_primary_metrics():
    frames = []
    for split_type in CONFIG["cell_splits"]:
        path = prediction_h5_path(split_type)
        require(path.exists(), f"Missing prediction HDF5: {path}")
        with h5py.File(path, "r") as h5:
            needed = {"cell_id", "observed_dge", "predicted_dge"}
            require(not needed.difference(h5.keys()),
                    f"{path} is missing datasets: {sorted(needed.difference(h5.keys()))}")
            obs = np.asarray(h5["observed_dge"], dtype=np.float32)
            pred = np.asarray(h5["predicted_dge"], dtype=np.float32)
            frame = pd.DataFrame({
                "split_type": split_type,
                "cell_id": decode_h5_strings(h5["cell_id"][:]),
                "delta_pcc": safe_pearson_rows(obs, pred, CONFIG["epsilon"]),
                "delta_r2": rowwise_r2(obs, pred),
                "dge_mse": ((obs - pred) ** 2).mean(axis=1),
                "naive_dge_mse": (obs ** 2).mean(axis=1),
            })
            frame["mse_gain_over_naive"] = frame["naive_dge_mse"] - frame["dge_mse"]
            frames.append(frame)

    rows = pd.concat(frames, ignore_index=True)
    metrics = (
        rows.groupby(["split_type", "cell_id"], as_index=False)[
            ["delta_pcc", "delta_r2", "dge_mse", "mse_gain_over_naive"]
        ].mean()
    )
    counts = (
        rows.groupby(["split_type", "cell_id"])
        .size().rename("n_test_perturbations").reset_index()
    )
    return metrics.merge(counts, on=["split_type", "cell_id"], how="left")

metrics_path = Path(CONFIG["a1_primary_metrics_path"])
if metrics_path.exists():
    primary_metrics = pd.read_csv(metrics_path)
    print(f"Loaded original Analysis 1A cell-level metrics: {metrics_path}")
else:
    print("Original Analysis 1A metric table not found; reconstructing from prediction HDF5 files.")
    primary_metrics = reconstruct_primary_metrics()

required_metric_cols = ["split_type", "cell_id", *METRIC_COLUMNS]
missing = [c for c in required_metric_cols if c not in primary_metrics.columns]
require(not missing, f"Cell-level performance table is missing columns: {missing}")

primary_metrics["split_type"] = primary_metrics["split_type"].astype(str)
primary_metrics["cell_id"] = primary_metrics["cell_id"].astype(str)
# Audit that every performance-table cell is a true test cell in its split,
# and that no training/test cell overlap exists.
for split_type in CONFIG["cell_splits"]:
    labels = adata.obs[split_type].astype(str)
    train_cells = set(
        adata.obs.loc[labels == "train", CONFIG["cell_column"]]
        .astype(str).unique()
    )
    test_cells_adata = set(
        adata.obs.loc[labels == "test", CONFIG["cell_column"]]
        .astype(str).unique()
    )
    test_cells_metrics = set(
        primary_metrics.loc[
            primary_metrics["split_type"] == split_type, "cell_id"
        ].astype(str).unique()
    )
    require(
        test_cells_metrics.issubset(test_cells_adata),
        f"{split_type}: performance table includes cells that are not test cells: "
        f"{sorted(test_cells_metrics.difference(test_cells_adata))}",
    )
    require(
        not train_cells.intersection(test_cells_metrics),
        f"{split_type}: training/test cell overlap detected.",
    )

save_table(primary_metrics, "cell_level_performance_metrics")
display(primary_metrics.head())

Loaded original Analysis 1A cell-level metrics: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/a1_ood_similarity/tables/a1_primary_cell_level_metrics.csv


,split_type,cell_id,delta_pcc,delta_r2,dge_mse,naive_dge_mse,mse_gain_over_naive,naive_delta_r2,delta_r2_gain_over_naive,n_test_perturbations,analysis_context,sparse_cell_warning,n_training_cells,n_test_cells_in_fold,nearest_training_cell,nearest_training_similarity_pearson,mean_top_3_training_similarity_pearson,nearest_training_similarity_mu_logvar_cosine,n_control_rows
0,cell_split1,CD34,0.540508,0.159890,1.113141,1.449615,0.336474,-3.403230e-13,0.159890,2739,observed_mixture,False,65,9,HL60,0.766298,0.759031,0.978579,257
1,cell_split1,JURKAT,0.495280,0.134738,1.734873,2.285584,0.550710,-1.988294e-13,0.134738,2469,observed_mixture,False,65,9,HL60,0.801608,0.788088,0.980301,100
2,cell_split1,MNEU.E,0.574885,0.326649,0.581849,0.879010,0.297161,-1.785099e-13,0.326649,634,observed_mixture,False,65,9,HELA,0.675522,0.651373,0.967539,66
3,cell_split1,NCIH2073,0.561861,0.289009,0.786125,1.115149,0.329024,-2.038452e-13,0.289009,717,observed_mixture,False,65,9,A549,0.949404,0.919204,0.983534,24
4,cell_split1,NCIH508,0.551331,0.305491,1.397684,1.846229,0.448545,-2.145860e-13,0.305491,689,observed_mixture,False,65,9,CL34,0.929249,0.892333,0.985908,16


## 6. Compute response similarity to every eligible training cell

In [7]:

# -----------------------------------------------------------------------------
# Pairwise similarity: arithmetic mean of condition-specific Pearson correlations.
# -----------------------------------------------------------------------------
ANALYSIS_NAME = "nearest_response_cell_similarity"
SIMILARITY_COLUMN_LABEL = "mean_conditionwise_dge_pearson"
SIMILARITY_DEFINITION = (
    "For each eligible test-training cell pair, compute ordinary Pearson "
    "correlation between the two 978-gene observed-DGE centroids separately "
    "for every exact shared drug-dose-duration condition; then take the "
    "arithmetic mean across usable matched conditions. The nearest training "
    "response cell maximizes that mean."
)

def cell_condition_lookup(cell_id):
    """
    Map an exact (drug_id, rounded dose, rounded duration) tuple to the row
    containing that cell-condition DGE centroid.

    Tuple keys are used instead of the display-only condition_key string.
    The previous string used general-format floating-point rendering, which
    could collapse distinct rounded numeric values to the same text label.
    """
    sub = condition_meta.loc[
        condition_meta["cell_id"] == str(cell_id)
    ].copy()

    tuple_keys = list(
        zip(
            sub["drug_key"].astype(str),
            sub["dose_key"].astype(float),
            sub["time_key"].astype(float),
        )
    )

    require(
        len(tuple_keys) == len(set(tuple_keys)),
        f"{cell_id}: duplicated exact drug-dose-duration tuple keys.",
    )

    return dict(zip(tuple_keys, sub["condition_row"].astype(int)))

lookup_cache = {
    cell: cell_condition_lookup(cell)
    for cell in condition_meta["cell_id"].unique()
}

pair_rows = []
condition_rows = []

for split_type in CONFIG["cell_splits"]:
    split_labels = adata.obs[split_type].astype(str)
    train_cells = sorted(
        adata.obs.loc[
            split_labels == "train", CONFIG["cell_column"]
        ].astype(str).unique()
    )
    test_cells = sorted(
        primary_metrics.loc[
            primary_metrics["split_type"] == split_type, "cell_id"
        ].astype(str).unique()
    )

    require(train_cells, f"{split_type}: no training cells.")
    require(test_cells, f"{split_type}: no test cells.")
    require(
        not set(train_cells).intersection(test_cells),
        f"{split_type}: training/test cell overlap.",
    )

    print(
        f"{split_type}: {len(train_cells)} training cells, "
        f"{len(test_cells)} test cells"
    )

    for test_cell in test_cells:
        test_lookup = lookup_cache.get(test_cell, {})
        require(test_lookup, f"No condition centroids found for test cell {test_cell}.")

        for train_cell in train_cells:
            train_lookup = lookup_cache.get(train_cell, {})
            require(
                train_lookup,
                f"No condition centroids found for training cell {train_cell}.",
            )

            shared = sorted(set(test_lookup).intersection(train_lookup))
            n_shared = len(shared)

            correlations = []
            condition_records = []
            if n_shared >= CONFIG["min_shared_perturbations"]:
                for condition in shared:
                    r = safe_pearson(
                        condition_dge[test_lookup[condition]],
                        condition_dge[train_lookup[condition]],
                        CONFIG["epsilon"],
                    )
                    if np.isfinite(r):
                        correlations.append(r)
                        drug_id, dose_value, time_value = condition
                        condition_records.append({
                            "split_type": split_type,
                            "test_cell": test_cell,
                            "training_cell": train_cell,
                            "drug_key": drug_id,
                            "dose_key": dose_value,
                            "time_key": time_value,
                            "condition_key": (
                                f"{drug_id}|dose={dose_value:.{CONFIG['dose_round_decimals']}f}"
                                f"|time={time_value:.{CONFIG['time_round_decimals']}f}"
                            ),
                            "condition_pearson": r,
                        })

            finite_corr = np.asarray(correlations, dtype=float)

            # Eligibility is based on USABLE matched correlations, not merely
            # nominal overlap. This prevents a pair with 30 shared conditions
            # but fewer than 30 estimable Pearson correlations from entering.
            eligible = (
                len(finite_corr) >= CONFIG["min_shared_perturbations"]
            )
            if eligible:
                condition_rows.extend(condition_records)

            pair_rows.append({
                "split_type": split_type,
                "test_cell": test_cell,
                "training_cell": train_cell,
                "n_shared_conditions": n_shared,
                "n_finite_condition_correlations": len(finite_corr),
                "eligible_min_shared": eligible,
                "mean_conditionwise_dge_pearson": (
                    float(finite_corr.mean()) if eligible else np.nan
                ),
                "median_conditionwise_dge_pearson": (
                    float(np.median(finite_corr)) if eligible else np.nan
                ),
                "sd_conditionwise_dge_pearson": (
                    float(finite_corr.std(ddof=1))
                    if eligible and len(finite_corr) > 1 else np.nan
                ),
                "q25_conditionwise_dge_pearson": (
                    float(np.quantile(finite_corr, 0.25))
                    if eligible else np.nan
                ),
                "q75_conditionwise_dge_pearson": (
                    float(np.quantile(finite_corr, 0.75))
                    if eligible else np.nan
                ),
            })

pairwise_similarity = pd.DataFrame(pair_rows)
condition_level_similarity = pd.DataFrame(condition_rows)

require(
    not pairwise_similarity.duplicated(
        ["split_type", "test_cell", "training_cell"]
    ).any(),
    "Duplicate test-training pair rows detected.",
)

save_table(
    pairwise_similarity,
    "response_similarity_all_test_training_pairs",
)
save_table(
    condition_level_similarity,
    "response_similarity_condition_level_correlations",
    save_csv=CONFIG["save_condition_level_csv"],
)

eligible_pairs = pairwise_similarity.loc[
    pairwise_similarity["eligible_min_shared"]
    & pairwise_similarity[SIMILARITY_COLUMN_LABEL].notna()
].copy()

require(
    not eligible_pairs.empty,
    "No eligible test-training cell pairs remain at the configured threshold.",
)

eligible_cell_keys = set(
    map(
        tuple,
        eligible_pairs[["split_type", "test_cell"]]
        .drop_duplicates().to_numpy(),
    )
)
all_test_cell_keys = set(
    map(
        tuple,
        primary_metrics[["split_type", "cell_id"]]
        .drop_duplicates().to_numpy(),
    )
)
missing_eligible = sorted(all_test_cell_keys.difference(eligible_cell_keys))
if missing_eligible:
    print(
        "WARNING: test cells without any eligible training-cell match "
        f"(threshold={CONFIG['min_shared_perturbations']}): {missing_eligible}"
    )

nearest_idx = (
    eligible_pairs.groupby(["split_type", "test_cell"])[
        SIMILARITY_COLUMN_LABEL
    ].idxmax()
)
nearest_similarity = (
    eligible_pairs.loc[nearest_idx]
    .rename(columns={
        "test_cell": "cell_id",
        "training_cell": "nearest_training_cell_name",
        SIMILARITY_COLUMN_LABEL: "nearest_training_cell_response",
        "n_shared_conditions": "nearest_n_shared_conditions",
    })
    [[
        "split_type",
        "cell_id",
        "nearest_training_cell_name",
        "nearest_training_cell_response",
        "nearest_n_shared_conditions",
        "n_finite_condition_correlations",
        "median_conditionwise_dge_pearson",
        "sd_conditionwise_dge_pearson",
        "q25_conditionwise_dge_pearson",
        "q75_conditionwise_dge_pearson",
    ]]
    .reset_index(drop=True)
)

require(
    not nearest_similarity.duplicated(["split_type", "cell_id"]).any(),
    "More than one nearest response cell was retained for a test cell.",
)

save_table(
    nearest_similarity,
    "response_similarity_nearest_training_cell",
)
display(
    nearest_similarity.sort_values(
        ["split_type", "nearest_training_cell_response"]
    )
)


cell_split1: 65 training cells, 9 test cells
cell_split2: 65 training cells, 9 test cells
cell_split3: 65 training cells, 9 test cells
cell_split4: 65 training cells, 9 test cells
cell_split5: 65 training cells, 9 test cells


,split_type,cell_id,nearest_training_cell_name,nearest_training_cell_response,nearest_n_shared_conditions,n_finite_condition_correlations,median_conditionwise_dge_pearson,sd_conditionwise_dge_pearson,q25_conditionwise_dge_pearson,q75_conditionwise_dge_pearson
2,cell_split1,MNEU.E,HS578T,0.073129,54,54,0.050616,0.108428,0.006181,0.095202
8,cell_split1,WSUDLCL2,SKM1,0.094750,347,347,0.065528,0.130391,0.008701,0.150647
1,cell_split1,JURKAT,U266,0.097287,238,238,0.076617,0.180983,-0.024324,0.166668
5,cell_split1,OV7,SKLU1,0.106720,355,355,0.096924,0.119172,0.027119,0.183498
6,cell_split1,SKL,SKB,0.148876,48,48,0.104360,0.229852,-0.005035,0.244522
3,cell_split1,NCIH2073,A549,0.152125,361,361,0.107173,0.175581,0.019966,0.249601
4,cell_split1,NCIH508,SW948,0.169285,357,357,0.149024,0.175783,0.048697,0.287676
0,cell_split1,CD34,HL60,0.191030,320,320,0.173909,0.212271,0.070076,0.314950
7,cell_split1,THP1,NOMO1,0.207175,631,631,0.180682,0.208882,0.057005,0.353190
14,cell_split2,RMUGS,SW620,0.081263,362,362,0.078966,0.158574,-0.017980,0.189341


## 7. Associate response similarity with unseen-cell prediction performance

In [8]:

# -----------------------------------------------------------------------------
# Merge response similarity with performance and run Analysis 1A-style associations.
# -----------------------------------------------------------------------------
analysis_cell_table = primary_metrics.merge(
    nearest_similarity,
    on=["split_type", "cell_id"],
    how="left",
    validate="one_to_one",
)

analysis_cell_table["eligible_response_similarity"] = (
    analysis_cell_table["nearest_training_cell_response"].notna()
)
save_table(
    analysis_cell_table,
    "response_similarity_cell_level_analysis",
)

coverage = (
    analysis_cell_table.groupby("split_type", as_index=False)
    .agg(
        n_test_cells=("cell_id", "nunique"),
        n_cells_with_eligible_training_match=(
            "eligible_response_similarity", "sum"
        ),
        min_nearest_shared_conditions=(
            "nearest_n_shared_conditions", "min"
        ),
        median_nearest_shared_conditions=(
            "nearest_n_shared_conditions", "median"
        ),
        max_nearest_shared_conditions=(
            "nearest_n_shared_conditions", "max"
        ),
    )
)
coverage["fraction_cells_eligible"] = (
    coverage["n_cells_with_eligible_training_match"]
    / coverage["n_test_cells"]
)
save_table(coverage, "response_similarity_coverage")
display(coverage)

association_rows = []
eligible_cells = analysis_cell_table.loc[
    analysis_cell_table["eligible_response_similarity"]
].copy()

for metric in METRIC_COLUMNS:
    for split_type in CONFIG["cell_splits"]:
        sub = eligible_cells.loc[
            eligible_cells["split_type"] == split_type
        ].copy()

        stat = bootstrap_fold_association(
            sub,
            x_col="nearest_training_cell_response",
            y_col=metric,
            n_bootstrap=CONFIG["n_bootstrap"],
            seed=deterministic_seed(
                CONFIG["random_seed"],
                ANALYSIS_NAME,
                split_type,
                metric,
            ),
        )
        association_rows.append({
            "analysis_name": ANALYSIS_NAME,
            "analysis_context": "observed_mixture",
            "similarity_metric": SIMILARITY_COLUMN_LABEL,
            "metric": metric,
            "split_type": split_type,
            **stat,
        })

fold_associations = pd.DataFrame(association_rows)
save_table(
    fold_associations,
    "response_similarity_fold_associations",
)

across_fold_summary = (
    fold_associations.groupby(
        ["analysis_name", "metric"], as_index=False
    )
    .agg(
        n_folds=("split_type", "nunique"),
        n_estimable_folds=("association_estimable", "sum"),
        mean_fold_spearman_rho=("spearman_rho", "mean"),
        sd_fold_spearman_rho=("spearman_rho", "std"),
        min_fold_spearman_rho=("spearman_rho", "min"),
        max_fold_spearman_rho=("spearman_rho", "max"),
    )
)
save_table(
    across_fold_summary,
    "response_similarity_across_fold_summary",
)

display(fold_associations.round(4))
display(across_fold_summary.round(4))


,split_type,n_test_cells,n_cells_with_eligible_training_match,min_nearest_shared_conditions,median_nearest_shared_conditions,max_nearest_shared_conditions,fraction_cells_eligible
0,cell_split1,9,9,48,347.0,631,1.0
1,cell_split2,9,9,49,352.0,688,1.0
2,cell_split3,9,9,31,66.0,618,1.0
3,cell_split4,9,9,32,354.0,742,1.0
4,cell_split5,9,9,65,352.0,689,1.0


,analysis_name,analysis_context,similarity_metric,metric,split_type,n_cells,n_bootstrap_requested,n_bootstrap_estimable_rho,n_bootstrap_estimable_slope,spearman_rho,spearman_p_descriptive,spearman_bootstrap_ci_low,spearman_bootstrap_ci_high,ols_slope,ols_intercept,ols_r2,ols_slope_bootstrap_ci_low,ols_slope_bootstrap_ci_high,association_estimable
0,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_pcc,cell_split1,9,2000,2000,2000,-0.3833,0.3085,-0.8957,0.3043,-0.2306,0.6066,0.0448,-1.0694,0.4631,True
1,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_pcc,cell_split2,9,2000,2000,2000,-0.0667,0.8647,-0.9474,0.7218,-0.1151,0.5947,0.0353,-1.0216,0.2764,True
2,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_pcc,cell_split3,9,2000,2000,2000,0.0833,0.8312,-0.7143,0.9450,0.0642,0.5543,0.0053,-0.6201,0.6412,True
3,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_pcc,cell_split4,9,2000,2000,2000,-0.2667,0.4879,-0.8948,0.5135,-0.4015,0.6533,0.1705,-1.3428,0.1370,True
4,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_pcc,cell_split5,9,2000,2000,2000,0.3833,0.3085,-0.3846,0.7949,1.3395,0.3106,0.1854,0.2056,4.0353,True
5,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_r2,cell_split1,9,2000,2000,2000,-0.4000,0.2861,-0.8947,0.3403,-0.7170,0.3917,0.1068,-2.1306,1.0314,True
6,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_r2,cell_split2,9,2000,2000,2000,-0.3667,0.3317,-1.0000,0.5263,-0.2190,0.3544,0.0520,-1.7049,0.4162,True
7,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_r2,cell_split3,9,2000,2000,2000,-0.0500,0.8984,-0.8929,0.7217,0.4337,0.2172,0.0648,-0.8713,1.5867,True
8,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_r2,cell_split4,9,2000,2000,2000,-0.4667,0.2054,-0.9455,0.3019,-0.7076,0.4426,0.2325,-2.1166,0.0265,True
9,nearest_response_cell_similarity,observed_mixture,mean_conditionwise_dge_pearson,delta_r2,cell_split5,9,2000,2000,2000,0.3333,0.3807,-0.5475,0.7143,8.4801,-1.4409,0.1130,-0.5718,30.0269,True


,analysis_name,metric,n_folds,n_estimable_folds,mean_fold_spearman_rho,sd_fold_spearman_rho,min_fold_spearman_rho,max_fold_spearman_rho
0,nearest_response_cell_similarity,delta_pcc,5,5,-0.0500,0.3016,-0.3833,0.3833
1,nearest_response_cell_similarity,delta_r2,5,5,-0.1900,0.3337,-0.4667,0.3333
2,nearest_response_cell_similarity,dge_mse,5,5,-0.0167,0.4933,-0.5000,0.5333
3,nearest_response_cell_similarity,mse_gain_over_naive,5,5,0.1800,0.3022,-0.1667,0.5000


## 8. Export manifest and finish

In [9]:
manifest = {
    "analysis_name": ANALYSIS_NAME,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": {
        key: str(value) if isinstance(value, Path) else value
        for key, value in CONFIG.items()
    },
    "similarity_definition": SIMILARITY_DEFINITION,
    "replicate_handling": (
        "Observed DGE profiles are averaged within each cell and exact "
        "drug-dose-duration condition before cross-cell comparison."
    ),
    "eligibility": (
        f"At least {CONFIG['min_shared_perturbations']} exact shared "
        "drug-dose-duration conditions per training-test cell pair."
    ),
    "downstream_association": (
        "Within each cell split, Spearman correlation across held-out cells "
        "between nearest response similarity and cell-level performance. "
        "Percentile 95% confidence intervals use cell-level bootstrap resampling."
    ),
    "diagnostic_only": (
        "Observed test-cell DGE is used to define response similarity. "
        "This is a post-hoc biological diagnostic, not a prospective DEPICT input."
    ),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scanpy": sc.__version__,
    },
}
(MANIFEST_DIR / "analysis_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str)
)
print(f"Completed {ANALYSIS_NAME}. Outputs: {OUT}")

Completed nearest_response_cell_similarity. Outputs: /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/Jun2026New/NoRetraining_Analysis/a1_response_similarity_mean_condition_v3
